In [2]:
# Cell 1: Imports + load
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Quick prep
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df = df.drop('customerID', axis=1)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Features & target
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [3]:
# Let's see which columns are categorical and which are numerical
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)
print("\nNumber of categorical features:", len(categorical_cols))
print("Number of numerical features:", len(numerical_cols))

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Number of categorical features: 15
Number of numerical features: 4


In [4]:
# Create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'  # if there are any other columns
)

print("Preprocessor created")

Preprocessor created


In [5]:
from sklearn.ensemble import RandomForestClassifier

# Full pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_split=5,
        random_state=42,
        class_weight='balanced',      # helps with the 73:27 imbalance
        n_jobs=-1
    ))
])

print("Random Forest pipeline ready")

Random Forest pipeline ready


In [6]:
# Fit on training data
rf_pipeline.fit(X_train, y_train)

print("Model training finished ✓")

Model training finished ✓


In [7]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Predict probabilities (for ROC-AUC)
y_pred_proba = rf_pipeline.predict_proba(X_test)[:, 1]

# Predict classes (for confusion matrix & report)
y_pred = rf_pipeline.predict(X_test)

# Evaluation
print("ROC AUC score:", round(roc_auc_score(y_test, y_pred_proba), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=3))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

ROC AUC score: 0.8386

Classification Report:

              precision    recall  f1-score   support

           0      0.872     0.807     0.838      1035
           1      0.557     0.671     0.608       374

    accuracy                          0.771      1409
   macro avg      0.714     0.739     0.723      1409
weighted avg      0.788     0.771     0.777      1409


Confusion Matrix:

[[835 200]
 [123 251]]


In [8]:
import joblib
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the trained pipeline
joblib.dump(rf_pipeline, '../models/churn_pipeline_rf.joblib')

print("Model saved successfully to ../models/churn_pipeline_rf.joblib")

Model saved successfully to ../models/churn_pipeline_rf.joblib


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

lr_pipeline.fit(X_train, y_train)

y_pred_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]
print("Logistic Regression ROC AUC:", round(roc_auc_score(y_test, y_pred_proba_lr), 4))